# Wizualizacja Uprawnień Looker - Wykres Sankey

Poniższy kod ładuje plik `sankey_data.json` i generuje piękny, interaktywny wykres przepływowy za pomocą biblioteki `Plotly`. 
Wykres prezentuje ścieżkę:
`Model -> Explore -> Dashboard -> Group -> User`

*Aby wykres działał, upewnij się że uruchomiłeś skrypty `extract_raw_data.py` oraz `build_sankey_data.py`.*

In [ ]:
# Wygeneruj sankey_data.json bez wychodzenia z Notatnika!
from build_sankey_data import build_sankey

build_sankey(
    input_file="permissions_looker_data.json",
    output_file="sankey_data.json",
    target_type="user",          # 'user' | 'group' | 'role'
    target_models=None,           # np. ["model_a", "model_b"]
    target_entities=None,         # np. ["jan.kowalski@firma.pl"] lub ["Nazwa Grupy"]
    target_explores=None,         # np. ["orders", "sessions"]
    target_dashboards=None        # np. ["Sales Overview"]
)


In [ ]:
import json, os
import plotly.graph_objects as go
from IPython.display import IFrame

with open('sankey_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

nodes = data.get('nodes', [])
links = data.get('links', [])

if not nodes or not links:
    print('Brak danych! Uruchom komórkę build_sankey powyżej.')
else:
    TYPE_COLORS = {
        'model':     'rgba(231,76,60,0.9)',
        'explore':   'rgba(230,126,34,0.9)',
        'dashboard': 'rgba(52,152,219,0.9)',
        'group':     'rgba(155,89,182,0.9)',
        'role':      'rgba(26,188,156,0.9)',
        'user':      'rgba(52,73,94,0.9)',
    }
    node_base_colors = [TYPE_COLORS.get(n.get('type'), 'rgba(150,150,150,0.85)') for n in nodes]
    link_base_colors = ['rgba(255,200,0,0.45)'] * len(links)

    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(
            pad=20, thickness=28,
            line=dict(color='white', width=0.5),
            label=[n.get('label','') for n in nodes],
            color=node_base_colors,
        ),
        link=dict(
            source=[l['source'] for l in links],
            target=[l['target'] for l in links],
            value=[l.get('value',1) for l in links],
            color=link_base_colors,
        )
    ))
    fig.update_layout(
        title_text='Przepływ Uprawnień: Model → Explore → Dashboard → Encja',
        font=dict(size=12, family='Inter, Arial, sans-serif', color='white'),
        height=820, paper_bgcolor='#1a1a2e',
        margin=dict(l=20, r=20, t=60, b=20)
    )

    # Eksport do HTML z wstrzykniętym JS do obsługi kliknięć
    html_str = fig.to_html(include_plotlyjs='cdn', full_html=True)

    click_js = """
<script>
document.addEventListener('DOMContentLoaded', function() {
  var gd = document.querySelector('.js-plotly-plot');
  if (!gd) return;

  var baseNodeColors = """ + json.dumps(node_base_colors) + """;
  var baseLinkColors = """ + json.dumps(link_base_colors) + """;
  var links = """ + json.dumps(links) + """;
  var DIMMED_NODE = 'rgba(200,200,200,0.1)';
  var DIMMED_LINK = 'rgba(200,200,200,0.04)';
  var ACTIVE_LINK = 'rgba(255,200,0,0.55)';
  var highlighted = false;

  gd.on('plotly_click', function(eventData) {
    var pt = eventData.points[0];
    if (pt.type !== 'sankey') return;

    // Reset na ponowne klikniecie
    if (highlighted) {
      Plotly.restyle(gd, {'node.color': [baseNodeColors], 'link.color': [baseLinkColors]});
      highlighted = false;
      return;
    }

    var clickedIdx = pt.index;
    var connLinks = new Set();
    var connNodes = new Set([clickedIdx]);

    links.forEach(function(l, i) {
      if (l.source === clickedIdx || l.target === clickedIdx) {
        connLinks.add(i);
        connNodes.add(l.source);
        connNodes.add(l.target);
      }
    });

    var newNodeColors = baseNodeColors.map(function(c, i) {
      return connNodes.has(i) ? c : DIMMED_NODE;
    });
    var newLinkColors = baseLinkColors.map(function(c, i) {
      return connLinks.has(i) ? ACTIVE_LINK : DIMMED_LINK;
    });

    Plotly.restyle(gd, {'node.color': [newNodeColors], 'link.color': [newLinkColors]});
    highlighted = true;
  });
});
</script>
"""
    # Wstrzyknij JS przed </body>
    html_str = html_str.replace('</body>', click_js + '</body>')

    out_path = os.path.join(os.getcwd(), 'sankey_chart.html')
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(html_str)

    print(f'💡 Kliknij węzeł aby wyróżnić ścieżki. Kliknij ponownie aby zresetować.')
    display(IFrame(src='sankey_chart.html', width='100%', height='870px'))
